# Práctica 5. Búsqueda por similaridad. 

## 0. Configuramos el entorno previo a la práctica. 

In [27]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import sys, os 
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent 
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "5°-practica" / "data" / 'archivo_emojis_Elfinanciero.csv'


device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [28]:
df = pd.read_csv(DATA_PATH)
titles = df["title"].dropna().head(1000).tolist()

print(f"Total de títulos cargados: {len(titles)}")
print("\nEjemplo de títulos:")
print(titles[:5])

Total de títulos cargados: 1000

Ejemplo de títulos:
['AMLO: otra duda sobre su cardenismo', 'Esto sabemos de la muerte de 39 migrantes en un incendio en Ciudad Juárez', 'Ricardo Monreal instala primer comité de proyecto de reconciliación en EU', 'Israel, en tiempo extra', 'Alfonso Herrera y su miedo tras tragedia de RBD en Brasil: ¿Qué es la enoclofobia?']


## 1. Cargamos el modelo y generamos los embeddings

In [29]:
def predict(texts, tokenizer, model, device, bs=128):
    """
    Obtiene embeddings para una lista de textos usando el modelo dado.
    """
    output = []
    for i in range(0, len(texts), bs):
        tokens = tokenizer(
            texts[i: i+bs],
            return_tensors="pt",
            padding='max_length',
            max_length=50,
            truncation=True
        )
        t = {
            "input_ids": tokens['input_ids'].to(device),
            "attention_mask": tokens['attention_mask'].to(device)
        }
        with torch.no_grad():
            pred = model(**t).last_hidden_state[:, 0].cpu()
        output.append(pred)
    output = torch.cat(output, dim=0)
    return output

## 2. Cargamos los modelos

In [30]:
#  Modelo 1: BETO
beto_path = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer_beto = AutoTokenizer.from_pretrained(beto_path)
model_beto = AutoModel.from_pretrained(beto_path).to(device)

#  Modelo 2: Robertuito 
robertuito_path = "pysentimiento/robertuito-base-uncased"
tokenizer_robertuito = AutoTokenizer.from_pretrained(robertuito_path)
model_robertuito = AutoModel.from_pretrained(robertuito_path).to(device)


Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at pysentimiento/robertuito-base-uncased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
# BETO
embeddings_beto = predict(titles, tokenizer_beto, model_beto, device)
print("Embeddings BETO:", embeddings_beto.shape)

# ROBERTUITO
embeddings_robertuito = predict(titles, tokenizer_robertuito, model_robertuito, device)
print("Embeddings Robertuito:", embeddings_robertuito.shape)


Embeddings BETO: torch.Size([1000, 768])
Embeddings Robertuito: torch.Size([1000, 768])


## 3. Búsqueda por similaridad

In [32]:
queries = [
    "Muerte de migrantes en un incendio en Ciudad Juárez",
    "¿Qué tan sano es comer tacos de cabeza?",
    "La crisis humanitaria de Estados Unidos",
    "Lo que hay del otro laredo",
    "¿Existe un acuerdo entre la banca tradicional y los nuevos jugadores?",
    "Maribel Guardia afirma: Ya no le tengo miedo",
    "Graban pelea en Xochimilco (Video)"
]

def search_similar(queries, titles, embeddings, tokenizer, model, top_k=5):
    """
    Dada una lista de consultas, devuelve los títulos más similares en base a coseno.
    """
    query_emb = predict(queries, tokenizer, model, device)
    sims = cosine_similarity(query_emb, embeddings)
    results = []
    
    for i, query in enumerate(queries):
        top_idx = np.argsort(sims[i])[::-1][:top_k]
        top_titles = [(titles[j], float(sims[i][j])) for j in top_idx]
        results.append({"query": query, "results": top_titles})
    
    return results


# Resultados BETO
results_beto = search_similar(queries, titles, embeddings_beto, tokenizer_beto, model_beto)

# Resultados ROBERTUITO
results_robertuito = search_similar(queries, titles, embeddings_robertuito, tokenizer_robertuito, model_robertuito)

## 4. Visualización y comparación

In [34]:
def display_results(results, model_name):
    print(f"\nResultados para {model_name}\n")
    for r in results:
        print(f" Consulta: {r['query']}")
        for title, score in r["results"]:
            print(f"    {title} (similitud={score:.4f})")

display_results(results_beto, "BETO")
display_results(results_robertuito, "Robertuito")


Resultados para BETO

 Consulta: Muerte de migrantes en un incendio en Ciudad Juárez
    Crisis migratoria: Detienen a un hombre que traficaba 93 migrantes en Nuevo León (similitud=0.9011)
    Muerte de migrantes en Ciudad Juárez: extitular del INM sugiere que los dejaron encerrados (similitud=0.9011)
    Esto sabemos de la muerte de 39 migrantes en un incendio en Ciudad Juárez (similitud=0.9008)
    Incendio en Ciudad Juárez: Revelan identidad de migrantes fallecidos en Chihuahua (similitud=0.8953)
    Incendio en Ciudad Juárez: Dan prisión preventiva a 5 acusados por muerte de migrantes (similitud=0.8894)
 Consulta: ¿Qué tan sano es comer tacos de cabeza?
    ¿Qué tanto daño hace comer tacos de tripa? (similitud=0.9641)
    ¿Qué tan sano es comer tacos de pastor? (similitud=0.9614)
    ¿Qué pasa si comes donas todos los días?  (similitud=0.9302)
    ¿Qué pasa si comes donas todos los días?  (similitud=0.9302)
    ¿Qué tan saludable es comer birria?  (similitud=0.9215)
 Consulta: La 

**Conclusión:**  
Al comparar los resultados de los modelos BETO y Robertuito, se observa que BETO responde mejor a similitudes léxicas, mientras que Robertuito capta similitudes semánticas más profundas.  
Por tanto, Robertuito ofrece resultados más coherentes en búsquedas contextuales, mostrando un mejor entendimiento del significado general de las consultas.
